In [1]:
import tensorflow as tf

In [2]:
 x = tf.Variable(3, name="x")
 y = tf.Variable(4, name="y")
 f = x*x*y + y + 2

In [4]:
# In TensorFlow 2.x, eager execution is enabled by default, so a session is not needed.
# The variables x and y are already initialized when they are created.

# evaluate f directly
result = f.numpy() # Use .numpy() to get the value of the tensor as a NumPy array
print(result)

42


In [6]:
x1 = tf.Variable(1)
# x1.graph is tf.get_default_graph() # This line is removed as it's not valid in TF 2.x eager execution

In [8]:
w = tf.constant(3)
x = w + 2
y = x + 5
z = x * 3
print(y.numpy())
print(z.numpy())

10
15


In [10]:
 y_val, z_val = y, z
 print(y_val)
 print(z_val)

tf.Tensor(10, shape=(), dtype=int32)
tf.Tensor(15, shape=(), dtype=int32)


### Linear Regression with Tensorflow

In [11]:
import numpy as np
import tensorflow as tf
from sklearn.datasets import fetch_california_housing

# Load and prepare data
housing = fetch_california_housing()
m, n = housing.data.shape
housing_data_plus_bias = np.c_[np.ones((m, 1)), housing.data]  # Add bias term (x0 = 1)

# Define TensorFlow constants (tensors)
X = tf.constant(housing_data_plus_bias, dtype=tf.float32, name="X")  # Shape: (m, n+1)
y = tf.constant(housing.target.reshape(-1, 1), dtype=tf.float32, name="y")  # Shape: (m, 1)

# Define normal equation: theta = (X^T X)^(-1) X^T y
XT = tf.transpose(X)  # Transpose: (n+1, m)
XTX = tf.matmul(XT, X)  # Matrix multiply: (n+1, m) * (m, n+1) = (n+1, n+1)
XTX_inv = tf.linalg.inv(XTX)  # Inverse: (n+1, n+1)
XTy = tf.matmul(XT, y)  # Matrix multiply: (n+1, m) * (m, 1) = (n+1, 1)
theta = tf.matmul(XTX_inv, XTy)  # Final theta: (n+1, 1)


 # Evaluate the graph to get theta
theta_value = theta.numpy()

print("Optimal parameters (theta):", theta_value)

Optimal parameters (theta): [[-3.7167969e+01]
 [ 4.3632507e-01]
 [ 9.3879700e-03]
 [-1.0710144e-01]
 [ 6.4526367e-01]
 [-4.1127205e-06]
 [-3.7808418e-03]
 [-4.2367554e-01]
 [-4.3707275e-01]]


In [12]:
# Predict on a test instance
X_test = housing_data_plus_bias[:5]  # First 5 instances
y_pred = np.matmul(X_test, theta_value)  # Compute predictions
print("Predictions:", y_pred.flatten())
print("Actual:", housing.target[:5])

Predictions: [4.12563547 3.97111782 3.67079548 3.23588801 2.40863961]
Actual: [4.526 3.585 3.521 3.413 3.422]


### Using Batch Gradient Descent

Gradient Descent requires scaling the feature vectors first. We could do this using TF, but let's just use Scikit-Learn for now.

In [13]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_housing_data = scaler.fit_transform(housing.data)
scaled_housing_data_plus_bias = np.c_[np.ones((m, 1)), scaled_housing_data]

In [14]:
print(scaled_housing_data_plus_bias.mean(axis=0))
print(scaled_housing_data_plus_bias.mean(axis=1))
print(scaled_housing_data_plus_bias.mean())
print(scaled_housing_data_plus_bias.shape)

[ 1.00000000e+00  6.60969987e-17  5.50808322e-18  6.60969987e-17
 -1.06030602e-16 -1.10161664e-17  3.44255201e-18 -1.07958431e-15
 -8.52651283e-15]
[ 0.38915536  0.36424355  0.5116157  ... -0.06612179 -0.06360587
  0.01359031]
0.11111111111111005
(20640, 9)


In [16]:
import numpy as np
import tensorflow as tf
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler

# Load and preprocess data
housing = fetch_california_housing()
m, n = housing.data.shape
scaler = StandardScaler()
scaled_housing_data = scaler.fit_transform(housing.data)  # Normalize features
scaled_housing_data_plus_bias = np.c_[np.ones((m, 1)), scaled_housing_data]  # Add bias term

# Hyperparameters
n_epochs = 1000
learning_rate = 0.01

# TensorFlow tensors
X = tf.constant(scaled_housing_data_plus_bias, dtype=tf.float32, name="X")  # Shape: (m, n+1)
y = tf.constant(housing.target.reshape(-1, 1), dtype=tf.float32, name="y")  # Shape: (m, 1)
theta = tf.Variable(tf.random.uniform([n + 1, 1], -1.0, 1.0), name="theta")  # Random init

# Training loop
for epoch in range(n_epochs):
    y_pred = tf.linalg.matmul(X, theta, name="predictions")  # X * theta
    error = y_pred - y  # Prediction error
    mse = tf.reduce_mean(tf.square(error), name="mse")  # MSE loss
    gradients = 2/m * tf.linalg.matmul(tf.transpose(X), error)  # Gradient: (2/m) * X^T * error
    theta.assign_sub(learning_rate * gradients)  # Update: theta -= learning_rate * gradients

    if epoch % 100 == 0:  # Print MSE every 100 epochs
        print(f"Epoch {epoch}, MSE = {mse.numpy():.6f}")

# Final parameters
best_theta = theta.numpy()
print("Optimal theta:", best_theta.flatten())

Epoch 0, MSE = 12.996080
Epoch 100, MSE = 0.945295
Epoch 200, MSE = 0.699596
Epoch 300, MSE = 0.652336
Epoch 400, MSE = 0.619925
Epoch 500, MSE = 0.596042
Epoch 600, MSE = 0.578355
Epoch 700, MSE = 0.565213
Epoch 800, MSE = 0.555414
Epoch 900, MSE = 0.548080
Optimal theta: [ 2.0685523   0.87349635  0.16619126 -0.2750825   0.28134325  0.01218514
 -0.0442748  -0.5128611  -0.4850751 ]


### Using auto diff for automatically calculating gradient

In [18]:
n_epochs = 1000
learning_rate = 0.01

# Initialize theta
theta = tf.Variable(tf.random.uniform([n + 1, 1], -1.0, 1.0), name="theta")  # Random init

# Define optimizer
optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)  # Gradient Descent optimizer

# Training loop with autodiff
for epoch in range(n_epochs):
    with tf.GradientTape() as tape:  # Track operations for autodiff
        y_pred = tf.linalg.matmul(X, theta)  # Predictions
        error = y_pred - y
        mse = tf.reduce_mean(tf.square(error))  # MSE loss

    gradients = tape.gradient(mse, [theta])[0]  # Autodiff computes gradient
    optimizer.apply_gradients([(gradients, theta)])  # Update theta

    if epoch % 100 == 0:  # Print MSE every 100 epochs
        print(f"Epoch {epoch}, MSE = {mse.numpy():.6f}")

# Final parameters
best_theta = theta.numpy()
print("Optimal theta:", best_theta.flatten())

Epoch 0, MSE = 9.438034
Epoch 100, MSE = 0.719372
Epoch 200, MSE = 0.565670
Epoch 300, MSE = 0.553219
Epoch 400, MSE = 0.545848
Epoch 500, MSE = 0.540444
Epoch 600, MSE = 0.536448
Epoch 700, MSE = 0.533482
Epoch 800, MSE = 0.531274
Epoch 900, MSE = 0.529624
Optimal theta: [ 2.0685523e+00  7.5340998e-01  1.2343372e-01 -8.6059272e-02
  1.4168797e-01 -1.8104108e-03 -3.8097456e-02 -9.3731976e-01
 -8.9738864e-01]


### Mini Batch Gradient Descent

In [20]:
# Hyperparameters
n_epochs = 10  # Reduced for demo
batch_size = 100
n_batches = int(np.ceil(m / batch_size))
learning_rate = 0.01

# Initialize theta
theta = tf.Variable(tf.random.uniform([n + 1, 1], -1.0, 1.0), name="theta")

# Optimizer
optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)

# Function to fetch mini-batch
def fetch_batch(epoch, batch_index, batch_size):
    np.random.seed(epoch * n_batches + batch_index)  # For reproducibility
    indices = np.random.randint(m, size=batch_size)
    # Convert NumPy indices to TensorFlow tensor
    tf_indices = tf.constant(indices, dtype=tf.int32)
    X_batch = tf.gather(X, tf_indices) # Use tf.gather for indexing with a tensor of indices
    y_batch = tf.gather(y, tf_indices) # Use tf.gather for indexing with a tensor of indices
    return X_batch, y_batch

# Training loop
for epoch in range(n_epochs):
    for batch_index in range(n_batches):
        X_batch, y_batch = fetch_batch(epoch, batch_index, batch_size)
        with tf.GradientTape() as tape:
            y_pred = tf.linalg.matmul(X_batch, theta)
            mse = tf.reduce_mean(tf.square(y_pred - y_batch))
        gradients = tape.gradient(mse, [theta])[0]
        optimizer.apply_gradients([(gradients, theta)])
    if epoch % 2 == 0:
        y_pred_full = tf.linalg.matmul(X, theta)
        mse_full = tf.reduce_mean(tf.square(y_pred_full - y))
        print(f"Epoch {epoch}, MSE = {mse_full.numpy():.6f}")

print("Optimal theta:", theta.numpy().flatten())

Epoch 0, MSE = 0.537862
Epoch 2, MSE = 0.532698
Epoch 4, MSE = 0.527411
Epoch 6, MSE = 0.526018
Epoch 8, MSE = 0.525227
Optimal theta: [ 2.0703032   0.85473067  0.11976179 -0.29794076  0.37459555  0.0034784
 -0.01152562 -0.86444    -0.832604  ]


### Modularity

In [24]:
import tensorflow as tf
import numpy as np

# Define modular ReLU function
def relu(X, name="relu"):
    with tf.name_scope(name):
        n_features = int(X.shape[1])  # Get number of features
        w = tf.Variable(tf.random.normal((n_features, 1)), name="weights")
        b = tf.Variable(0.0, name="bias")
        z = tf.linalg.matmul(X, w) + b
        return tf.maximum(z, 0.0, name="relu")

# Small dataset
X = tf.constant([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=tf.float32)  # Shape: (2, 3)

# Create 5 ReLUs
relus = [relu(X, name=f"relu_{i}") for i in range(5)]
output = tf.add_n(relus, name="output")